# OCCTO ユニット別発電実績 分析ノートブック

`dlh_dev` カタログの `silver.occto_unit_generation_actuals` を DuckDB `iceberg_scan()` 経由でクエリし、エリア別・発電方式別・発電所別の切り口で発電実績を可視化する。

テーブルは「発電所 × ユニット × 対象日 × 30分コマ（`time_code` 1〜48, 開始時刻基準）」のlong 粒度。詳細は `docs/tasks/plan_occto_pipeline.md` を参照。

> **DuckDB クエリ方式**: 約1,970万行を一括ロードせず、各セルが SQL 集計クエリを発行して小さい集計結果のみ受け取る。Iceberg パーティションプルーニングが自動的に効く。

## 1. セットアップ

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import plotly.express as px
import polars as pl

WORKSPACE_ROOT = (
    Path.cwd().resolve().parents[1] if Path.cwd().name == "Jupyter" else Path("/workspace")
)
SRC_PATH = WORKSPACE_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from common.duckdb_utils import create_duckdb_connection

pl.Config.set_tbl_rows(20)

## 2. DuckDB 接続とテーブルビューの作成

`create_duckdb_connection()` で S3 (RustFS) 設定済み DuckDB コネクションを取得し、
`iceberg_scan()` を指す `occto` ビューを作成する。
全行を Polars にロードせず、各セルが `conn.execute(SQL)` で必要な集計結果のみ取得する。

In [ ]:
TABLE_LOC = "s3://jp-power-grid-dev/silver/occto_unit_generation_actuals"

conn = create_duckdb_connection()
conn.execute(f"CREATE OR REPLACE VIEW occto AS SELECT * FROM iceberg_scan('{TABLE_LOC}')")

(total_rows,) = conn.execute("SELECT COUNT(*) FROM occto").fetchone()
print(f"total_rows={total_rows:,}")

## 3. スキーマと期間の確認

In [ ]:
conn.execute("DESCRIBE occto").pl()

In [ ]:
conn.execute("""
    SELECT
        MIN(target_date) AS min_target_date,
        MAX(target_date) AS max_target_date
    FROM occto
""").pl()

## 4. データ品質チェック

欠損値の件数と、発電所・ユニット・エリア・発電方式の件数を確認する。`generation_kwh` の欠損は、DuckDB の `UNPIVOT` 段階で「その日その発電所のコマの一部だけが欠測」だったケースに相当する（詳細は `bronze_to_silver_occto_unit_generation_actuals.py` 参照）。

In [ ]:
conn.execute("""
    SELECT
        COUNT(*) FILTER (WHERE power_plant_code IS NULL)                      AS power_plant_code,
        COUNT(*) FILTER (WHERE unit_name IS NULL)                             AS unit_name,
        COUNT(*) FILTER (WHERE target_date IS NULL)                           AS target_date,
        COUNT(*) FILTER (WHERE time_code IS NULL)                             AS time_code,
        COUNT(*) FILTER (WHERE generation_kwh IS NULL)                        AS generation_kwh,
        COUNT(*) FILTER (WHERE area IS NULL)                                  AS area,
        COUNT(*) FILTER (WHERE power_plant_name IS NULL)                      AS power_plant_name,
        COUNT(*) FILTER (WHERE power_generation_method_and_fuel_type IS NULL) AS power_generation_method_and_fuel_type
    FROM occto
""").pl()

In [ ]:
n_plants, n_areas, n_methods = conn.execute("""
    SELECT
        COUNT(DISTINCT power_plant_code)                      AS n_plants,
        COUNT(DISTINCT area)                                  AS n_areas,
        COUNT(DISTINCT power_generation_method_and_fuel_type) AS n_methods
    FROM occto
""").fetchone()
n_units = conn.execute(
    "SELECT COUNT(*) FROM (SELECT DISTINCT power_plant_code, unit_name FROM occto)"
).fetchone()[0]

print(f"発電所数: {n_plants}")
print(f"発電所×ユニット数: {n_units}")
print(f"エリア数: {n_areas}")
print(f"発電方式数: {n_methods}")

## 5. エリア別 総発電量

対象期間全体の `generation_kwh` をエリアごとに合計する（GWh 換算）。

In [ ]:
area_total = conn.execute("""
    SELECT area, SUM(generation_kwh) / 1e6 AS total_gwh
    FROM occto
    GROUP BY area
    ORDER BY total_gwh DESC
""").pl()

fig = px.bar(
    area_total,
    x="area",
    y="total_gwh",
    title="エリア別 総発電量",
    labels={"area": "エリア", "total_gwh": "総発電量 (GWh)"},
)
fig.show()

## 6. 発電方式別 総発電量

`power_generation_method_and_fuel_type` ごとに合計する。

In [ ]:
method_total = conn.execute("""
    SELECT power_generation_method_and_fuel_type, SUM(generation_kwh) / 1e6 AS total_gwh
    FROM occto
    GROUP BY power_generation_method_and_fuel_type
    ORDER BY total_gwh DESC
""").pl()

fig = px.bar(
    method_total,
    x="power_generation_method_and_fuel_type",
    y="total_gwh",
    title="発電方式別 総発電量",
    labels={
        "power_generation_method_and_fuel_type": "発電方式・燃種",
        "total_gwh": "総発電量 (GWh)",
    },
)
fig.show()

## 7. 発電所別 総発電量ランキング（上位20）

発電所は全国で301箇所あるため、まず上位20箇所を横棒グラフで比較する。以降のセルはここで作るランキングを土台にする。

In [ ]:
TOP_N = 20

plant_ranking = conn.execute("""
    SELECT power_plant_code, power_plant_name, area, SUM(generation_kwh) / 1e6 AS total_gwh
    FROM occto
    GROUP BY power_plant_code, power_plant_name, area
    ORDER BY total_gwh DESC
""").pl()

top_plants = plant_ranking.head(TOP_N)

fig = px.bar(
    top_plants.sort("total_gwh"),
    x="total_gwh",
    y="power_plant_name",
    color="area",
    orientation="h",
    title=f"発電所別 総発電量ランキング（上位{TOP_N}）",
    labels={
        "power_plant_name": "発電所名",
        "total_gwh": "総発電量 (GWh)",
        "area": "エリア",
    },
    height=600,
)
fig.show()

## 8. 日次総発電量の推移（全国合計）

全発電所を合計した日次発電量の時系列。データの欠落や異常値がないかの全体感の確認も兼ねる。

In [ ]:
daily_total = conn.execute("""
    SELECT target_date, SUM(generation_kwh) / 1e6 AS total_gwh
    FROM occto
    GROUP BY target_date
    ORDER BY target_date
""").pl()

fig = px.line(
    daily_total,
    x="target_date",
    y="total_gwh",
    title="日次総発電量の推移（全国合計）",
    labels={"target_date": "対象日", "total_gwh": "総発電量 (GWh)"},
)
fig.show()

## 9. 上位発電所の日次発電量推移

セクション7で求めた上位5発電所について、日次発電量の推移を重ねて比較する。

In [ ]:
TOP_N_TREND = 5
top_plant_codes = top_plants.head(TOP_N_TREND)["power_plant_code"].to_list()

daily_by_top_plant = conn.execute("""
    SELECT target_date, power_plant_name, SUM(generation_kwh) / 1e3 AS total_mwh
    FROM occto
    WHERE power_plant_code = ANY(?)
    GROUP BY target_date, power_plant_name
    ORDER BY target_date
""", [top_plant_codes]).pl()

fig = px.line(
    daily_by_top_plant,
    x="target_date",
    y="total_mwh",
    color="power_plant_name",
    title=f"上位{TOP_N_TREND}発電所の日次発電量推移",
    labels={
        "target_date": "対象日",
        "total_mwh": "日次発電量 (MWh)",
        "power_plant_name": "発電所名",
    },
)
fig.show()

## 10. 発電所別 1日の発電カーブ（コマ別平均）

上位3発電所について、`time_code`（30分コマ、開始時刻基準）ごとに全期間の平均発電量を求め、典型的な1日の発電パターンを比較する。原子力・石炭火力は日内でほぼ一定（ベースロード）になりやすく、水力などは変動が大きくなりやすい。

In [ ]:
TOP_N_PROFILE = 3
profile_plant_codes = top_plants.head(TOP_N_PROFILE)["power_plant_code"].to_list()


def slot_label(time_code: int) -> str:
    """time_code は開始時刻基準（1 -> 00:00, 48 -> 23:30）。"""
    total_minutes = (time_code - 1) * 30
    hour, minute = divmod(total_minutes, 60)
    return f"{hour % 24:02d}:{minute:02d}"


SLOT_ORDER = [slot_label(tc) for tc in range(1, 49)]

intraday_profile = conn.execute("""
    SELECT power_plant_name, time_code, AVG(generation_kwh) AS avg_generation_kwh
    FROM occto
    WHERE power_plant_code = ANY(?)
    GROUP BY power_plant_name, time_code
    ORDER BY power_plant_name, time_code
""", [profile_plant_codes]).pl().with_columns(
    pl.col("time_code")
    .map_elements(slot_label, return_dtype=pl.Utf8)
    .alias("slot_start")
)

fig = px.line(
    intraday_profile,
    x="slot_start",
    y="avg_generation_kwh",
    color="power_plant_name",
    category_orders={"slot_start": SLOT_ORDER},
    title="発電所別 1日の発電カーブ（コマ別平均, 全期間平均）",
    labels={
        "slot_start": "時刻（コマ開始時刻）",
        "avg_generation_kwh": "平均発電量 (kWh/30分)",
        "power_plant_name": "発電所名",
    },
)
fig.show()

## 11. 個別発電所の詳細確認

`SELECTED_PLANT_CODE` を変更すれば任意の発電所（全301箇所）の実績を確認できる。既定値はセクション7の1位（総発電量が最大の発電所）。

In [ ]:
# 発電所コードを変更すると、その発電所の実績に切り替わる
SELECTED_PLANT_CODE = top_plants[0, "power_plant_code"]

plant_meta = conn.execute("""
    SELECT power_plant_name, area, power_generation_method_and_fuel_type
    FROM occto
    WHERE power_plant_code = ?
    LIMIT 1
""", [SELECTED_PLANT_CODE]).fetchone()
plant_name, plant_area, plant_method = plant_meta

print(f"発電所: {plant_name}（{SELECTED_PLANT_CODE}）")
print(f"エリア: {plant_area} / 発電方式: {plant_method}")

In [ ]:
plant_daily = conn.execute("""
    SELECT target_date, SUM(generation_kwh) / 1e3 AS total_mwh
    FROM occto
    WHERE power_plant_code = ?
    GROUP BY target_date
    ORDER BY target_date
""", [SELECTED_PLANT_CODE]).pl()

fig = px.line(
    plant_daily,
    x="target_date",
    y="total_mwh",
    title=f"{plant_name} の日次発電量推移",
    labels={"target_date": "対象日", "total_mwh": "日次発電量 (MWh)"},
)
fig.show()

In [ ]:
plant_intraday = conn.execute("""
    SELECT
        time_code,
        AVG(generation_kwh)    AS avg_generation_kwh,
        STDDEV(generation_kwh) AS std_generation_kwh
    FROM occto
    WHERE power_plant_code = ?
    GROUP BY time_code
    ORDER BY time_code
""", [SELECTED_PLANT_CODE]).pl().with_columns(
    pl.col("time_code")
    .map_elements(slot_label, return_dtype=pl.Utf8)
    .alias("slot_start")
)

fig = px.line(
    plant_intraday,
    x="slot_start",
    y="avg_generation_kwh",
    error_y="std_generation_kwh",
    category_orders={"slot_start": SLOT_ORDER},
    title=f"{plant_name} の1日の発電カーブ（コマ別平均 ± 標準偏差）",
    labels={
        "slot_start": "時刻（コマ開始時刻）",
        "avg_generation_kwh": "平均発電量 (kWh/30分)",
    },
)
fig.show()